# SuperKart Sales Forecast with MLOps Pipeline

## Business Context
A sales forecast predicts future sales revenue based on historical data, industry trends, and the status of the current sales pipeline. This notebook builds an automated machine learning pipeline to predict product sales totals using store and product attributes.

## Objective
The goal is to build a reusable MLOps pipeline that loads data, cleans it, splits it, trains a model, evaluates performance, and prepares the result for deployment.

## Data Dictionary
- Product_Id: Unique identifier of each product
- Product_Weight: Weight of the product
- Product_Sugar_Content: Low Sugar / Regular / No Sugar
- Product_Allocated_Area: Allocated display area ratio
- Product_Type: Product category
- Product_MRP: Maximum retail price
- Store_Id: Unique identifier of each store
- Store_Establishment_Year: Year the store was established
- Store_Size: Small / Medium / High
- Store_Location_City_Type: Tier 1 / Tier 2 / Tier 3
- Store_Type: Store format type
- Product_Store_Sales_Total: Target variable; total sales revenue for the product in that store

## Expected output
This notebook will produce a cleaned dataset, model metrics, saved model, and a deployment-ready workflow for GitHub and Hugging Face.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path.cwd()
candidate_roots = [project_root, project_root.parent]
for candidate in candidate_roots:
    raw_path = candidate / 'data' / 'raw' / 'SuperKart.csv'
    if raw_path.exists():
        project_root = candidate
        break

raw_path = project_root / 'data' / 'raw' / 'SuperKart.csv'
processed_dir = project_root / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)
print(f'Project root: {project_root}')
print(f'Raw data path: {raw_path}')
print(f'Raw file exists: {raw_path.exists()}')

In [ ]:
df = pd.read_csv(raw_path)
print('Shape:', df.shape)
print('Columns:', list(df.columns))
print(df.head())
print('Missing values:\n', df.isnull().sum())
print('Duplicated rows:', df.duplicated().sum())
print(df.dtypes)

## Data Cleaning
The dataset has no missing values, but we still perform cleaning to remove unnecessary identifiers and ensure data quality before model training.

In [ ]:
clean_df = df.copy()

# Remove unnecessary identifier columns
identifier_cols = ['Product_Id', 'Store_Id']
clean_df = clean_df.drop(columns=[c for c in identifier_cols if c in clean_df.columns], errors='ignore')

# Ensure target column exists
target_col = 'Product_Store_Sales_Total'
if target_col not in clean_df.columns:
    raise ValueError(f'Target column {target_col} not found in dataset')

# Fill missing values properly
for column in clean_df.columns:
    if pd.api.types.is_numeric_dtype(clean_df[column]):
        clean_df[column] = clean_df[column].fillna(clean_df[column].median())
    else:
        mode_value = clean_df[column].mode(dropna=True)
        if not mode_value.empty:
            clean_df[column] = clean_df[column].fillna(mode_value.iloc[0])
        else:
            clean_df[column] = clean_df[column].fillna('Unknown')

# Remove duplicate rows
clean_df = clean_df.drop_duplicates().reset_index(drop=True)
print('Cleaned shape:', clean_df.shape)
print(clean_df.isnull().sum())

In [ ]:
from sklearn.model_selection import train_test_split

X = clean_df.drop(columns=[target_col])
y = clean_df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

train_df = X_train.copy()
train_df[target_col] = y_train
test_df = X_test.copy()
test_df[target_col] = y_test

train_df.to_csv(processed_dir / 'train.csv', index=False)
test_df.to_csv(processed_dir / 'test.csv', index=False)

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
print('Files saved to:', processed_dir)

## Model Training and Evaluation
A Random Forest Regressor is used because the target variable is continuous sales total and the model works well with mixed tabular data.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

train_df = pd.read_csv(processed_dir / 'train.csv')
test_df = pd.read_csv(processed_dir / 'test.csv')

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]
X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)
X_train, X_test = X_train.align(X_test, join='outer', axis=1, fill_value=0)

model = RandomForestRegressor(n_estimators=300, random_state=42)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = mean_squared_error(y_test, predictions, squared=False)
r2 = r2_score(y_test, predictions)

print('Mean Absolute Error (MAE):', round(mae, 2))
print('Root Mean Squared Error (RMSE):', round(rmse, 2))
print('R2 Score:', round(r2, 4))

model_dir = project_root / 'models'
model_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(model, model_dir / 'best_model.pkl')
print('Model saved at:', model_dir / 'best_model.pkl')

## Model Registration and Deployment Hints

### Hugging Face workflow
1. Create a dataset repository on Hugging Face and upload `train.csv` and `test.csv`.
2. Create a model repository on Hugging Face and upload the saved model file `best_model.pkl`.
3. Create a Hugging Face Space and select the Streamlit template.
4. Push the deployment files and app code to the Space.
5. Add the public Hugging Face Space URL to this notebook before final submission.

### GitHub workflow
1. Create a GitHub repository.
2. Upload the notebook, scripts, Dockerfile, and workflow files.
3. Add `.github/workflows/pipeline.yml` to automate the training process.
4. Add the GitHub repository link to this notebook before final submission.

### Deployment note
The final app should load the saved model from Hugging Face, accept user inputs, create a DataFrame, generate a prediction, and render the result with Streamlit.

## Final Submission Checklist
- [ ] Notebook is completed end-to-end
- [ ] All code cells run without error
- [ ] Data cleaning and train/test split are visible
- [ ] Model training and evaluation outputs are visible
- [ ] Model is saved locally
- [ ] GitHub repository link is added
- [ ] Hugging Face Space link is added
- [ ] Notebook is exported to HTML before submission

## Business Insight
The model helps the business estimate future sales totals, improve forecasting reliability, optimize procurement planning, and support smarter decision-making across stores and regions. The final deployment can help different business verticals adjust inventory and market strategy with more confidence.